# Data API Collector — Databricks Integration Example

This notebook demonstrates how to connect to the Data API Collector stack from Databricks:
1. **Kafka Streaming** — Read real-time synthetic data (fraud, telemetry, web traffic)
2. **Neo4j** — Query the graph database via Bolt
3. **PostgreSQL** — Read tables via JDBC
4. **REST API** — Interact with the API endpoints

## Prerequisites
- Data API Collector stack running with generators active
- Network access from Databricks to your host (nginx TCP forwarding on ports 9093, 7688)
- Databricks secrets configured (see Setup cell below)

In [ ]:
%pip install neo4j
dbutils.library.restartPython()

## Setup — Secrets & Configuration

Store your credentials using Databricks secrets before running this notebook:

```bash
databricks secrets create-scope data-api
databricks secrets put-secret data-api api-key --string-value "YOUR_SECRET_KEY"
databricks secrets put-secret data-api kafka-user --string-value "YOUR_KAFKA_USER"
databricks secrets put-secret data-api kafka-password --string-value "YOUR_KAFKA_PASSWORD"
databricks secrets put-secret data-api neo4j-password --string-value "YOUR_NEO4J_PASSWORD"
databricks secrets put-secret data-api postgres-password --string-value "YOUR_POSTGRES_PASSWORD"
```

In [ ]:
import neo4j
import requests
from pyspark.sql.types import *
from pyspark.sql.functions import *

# ---------------------------------------------------------------------------
# Configuration — UPDATE THESE for your environment
# ---------------------------------------------------------------------------
HOST = "your-hostname.com"                  # your nginx host

# Secrets — loaded from Databricks secret scope
API_KEY         = dbutils.secrets.get(scope="data-api", key="api-key")
KAFKA_USER      = dbutils.secrets.get(scope="data-api", key="kafka-user")
KAFKA_PASSWORD  = dbutils.secrets.get(scope="data-api", key="kafka-password")
NEO4J_USER      = "neo4j"
NEO4J_PASSWORD  = dbutils.secrets.get(scope="data-api", key="neo4j-password")
POSTGRES_USER   = "postgres"
POSTGRES_PASSWORD = dbutils.secrets.get(scope="data-api", key="postgres-password")

# Derived URLs
API_URL       = f"https://{HOST}/api/v1"
KAFKA_BROKER  = f"{HOST}:9093"
NEO4J_URI     = f"bolt+s://{HOST}:7688"
POSTGRES_JDBC = f"jdbc:postgresql://{HOST}:15433/datacollector?ssl=true&sslmode=require"
HEADERS       = {"X-Api-Key": API_KEY}

# Checkpoint base path — update to your catalog/schema
CHECKPOINT_BASE = "/Volumes/your_catalog/your_schema/checkpoints"

---
## 1. REST API — Start Kafka Generators

Start the streaming data generators on the server. These produce synthetic data to Kafka topics.

In [ ]:
# Start all three generators
for use_case in ["fraud_detection", "telemetry", "web_traffic"]:
    resp = requests.post(f"{API_URL}/kafka/generators/start", headers=HEADERS, json={
        "use_case": use_case,
        "rows_per_batch": 100,
        "batch_interval_seconds": 1.0,
        "timeout_minutes": 10,
    })
    data = resp.json()
    print(f"{use_case}: id={data.get('generator_id')} status={data.get('status')}")

# Check running generators
generators = requests.get(f"{API_URL}/kafka/generators", headers=HEADERS).json()
print(f"\n{len(generators)} generator(s) running")

---
## 2. Kafka Streaming — Fraud Detection

Read the `streaming-fraud-transactions` topic as a Structured Streaming DataFrame.

In [ ]:
fraud_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("merchant_id", IntegerType()),
    StructField("amount", DecimalType(10, 2)),
    StructField("currency", StringType()),
    StructField("merchant_category", StringType()),
    StructField("payment_method", StringType()),
    StructField("ip_address", StringType()),
    StructField("device_id", StringType()),
    StructField("latitude", DecimalType(8, 5)),
    StructField("longitude", DecimalType(9, 5)),
    StructField("card_type", StringType()),
    StructField("event_timestamp", StringType()),
])

raw_fraud = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-fraud-transactions")
    .option("startingOffsets", "earliest")
    .load()
)

fraud_parsed = (
    raw_fraud
    .select(from_json(col("value").cast("string"), fraud_schema).alias("data"))
    .select("data.*")
)

In [ ]:
# Preview the fraud stream — clear checkpoint first if restarting
dbutils.fs.rm(f"{CHECKPOINT_BASE}/fraud_preview", recurse=True)
display(fraud_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/fraud_preview")

### Write fraud stream to Delta table (optional)

In [ ]:
# Uncomment to write the fraud stream to a Delta table
# (
#     fraud_parsed.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("checkpointLocation", f"{CHECKPOINT_BASE}/fraud_transactions")
#     .toTable("your_catalog.your_schema.fraud_transactions")
# )

---
## 3. Kafka Streaming — Telemetry

Read the `streaming-device-telemetry` topic.

In [ ]:
telemetry_schema = StructType([
    StructField("event_id", StringType()),
    StructField("device_id", StringType()),
    StructField("device_type", StringType()),
    StructField("reading_value", DecimalType(10, 4)),
    StructField("unit", StringType()),
    StructField("battery_level", DecimalType(5, 2)),
    StructField("signal_strength_dbm", IntegerType()),
    StructField("firmware_version", StringType()),
    StructField("facility_id", IntegerType()),
    StructField("latitude", DecimalType(8, 5)),
    StructField("longitude", DecimalType(9, 5)),
    StructField("anomaly_flag", BooleanType()),
    StructField("event_timestamp", StringType()),
])

raw_telemetry = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-device-telemetry")
    .option("startingOffsets", "earliest")
    .load()
)

telemetry_parsed = (
    raw_telemetry
    .select(from_json(col("value").cast("string"), telemetry_schema).alias("data"))
    .select("data.*")
)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/telemetry_preview", recurse=True)
display(telemetry_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/telemetry_preview")

---
## 4. Kafka Streaming — Web Traffic

Read the `streaming-web-traffic` topic.

In [ ]:
web_schema = StructType([
    StructField("event_id", StringType()),
    StructField("session_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("page_url", StringType()),
    StructField("referrer", StringType()),
    StructField("user_agent", StringType()),
    StructField("ip_address", StringType()),
    StructField("country", StringType()),
    StructField("device_type", StringType()),
    StructField("action", StringType()),
    StructField("duration_ms", IntegerType()),
    StructField("http_status", IntegerType()),
    StructField("event_timestamp", StringType()),
])

raw_web = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-web-traffic")
    .option("startingOffsets", "earliest")
    .load()
)

web_parsed = (
    raw_web
    .select(from_json(col("value").cast("string"), web_schema).alias("data"))
    .select("data.*")
)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/web_preview", recurse=True)
display(web_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/web_preview")

---
## 5. Neo4j — Graph Queries

Query the Neo4j graph database over Bolt (TLS via nginx on port 7688).

In [ ]:
driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query):
    """Execute a Cypher query and return results as a list of dicts."""
    with driver.session() as session:
        result = session.run(query)
        return [record.data() for record in result]

# Test connectivity
print(run_cypher("RETURN 1 AS test"))

# Get database stats
stats = run_cypher("""
    CALL db.labels() YIELD label
    RETURN label, count(*) AS count
""")
print(f"\nLabels: {stats}")

In [ ]:
# Example: query nodes and convert to Spark DataFrame
results = run_cypher("""
    MATCH (n)
    RETURN labels(n) AS labels, count(n) AS count
""")

if results:
    df_neo4j = spark.createDataFrame(results)
    display(df_neo4j)
else:
    print("No nodes in database yet")

---
## 6. PostgreSQL — JDBC Read

Read tables from the PostgreSQL database. Requires the PostgreSQL JDBC driver on your cluster (typically pre-installed on Databricks).

In [ ]:
# PostgreSQL with native SSL on port 15433
df_events = (
    spark.read
    .format("jdbc")
    .option("url", POSTGRES_JDBC)
    .option("dbtable", "kafka_event_logs")
    .option("user", POSTGRES_USER)
    .option("password", POSTGRES_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

display(df_events)

---
## 7. REST API — Service Health Checks

Test connectivity to all services through the API.

In [ ]:
checks = {
    "PostgreSQL ORM":  "/data-sources/orm",
    "PostgreSQL SQL":  "/data-sources/raw/sql",
    "Neo4j Health":    "/data-sources/neo4j/health",
    "Neo4j Version":   "/data-sources/neo4j/version",
    "Redis":           "/redis/test",
    "Kafka Events":    "/kafka/events?limit=1",
}

for name, path in checks.items():
    try:
        r = requests.get(f"{API_URL}{path}", headers=HEADERS, timeout=10)
        status = r.json().get("status", r.status_code)
        print(f"  {name}: {status}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

---
## 8. Cleanup

Stop all running generators and close connections.

In [ ]:
# Stop all running generators
generators = requests.get(f"{API_URL}/kafka/generators", headers=HEADERS).json()
for g in generators:
    if g["status"] == "running":
        requests.post(f"{API_URL}/kafka/generators/{g['generator_id']}/stop", headers=HEADERS)
        print(f"Stopped {g['use_case']} generator {g['generator_id']}")

# Clean up finished generators
cleanup = requests.delete(f"{API_URL}/kafka/generators/cleanup", headers=HEADERS).json()
print(f"Cleaned up {cleanup.get('removed', 0)} generator(s)")

# Close Neo4j driver
driver.close()
print("Neo4j driver closed")

# Clear all checkpoints (optional)
# for cp in ["fraud_preview", "telemetry_preview", "web_preview"]:
#     dbutils.fs.rm(f"{CHECKPOINT_BASE}/{cp}", recurse=True)
# print("Checkpoints cleared")